# Mental Health Risk Model Training

**Novelle — AI-Powered Maternal Health Risk Support Platform**

This notebook trains the mental health risk prediction model using XGBoost.

## Model Overview
- **Target**: Mental health risk level (LOW / MEDIUM / HIGH)
- **Input Features**: PHQ-9, GAD-7, mood, stress, social support scores
- **Algorithm**: XGBoost Classifier with hyperparameter tuning
- **Explainability**: SHAP values for feature importance

---

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy scikit-learn xgboost lightgbm shap imbalanced-learn matplotlib seaborn joblib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, roc_auc_score, precision_score, recall_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

print("✅ Libraries loaded successfully")
print(f"   XGBoost version: {xgb.__version__}")

## 2. Load Data

In [ ]:
# Paths
DATA_DIR = Path('../datasets')
MODEL_DIR = Path('../../backend/app/ml/models')
REPORT_DIR = Path('../reports')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
mental_df = pd.read_csv(DATA_DIR / 'synthetic_mental_health.csv')
profiles_df = pd.read_csv(DATA_DIR / 'synthetic_profiles.csv')

print(f"Mental health records: {len(mental_df):,}")
print(f"User profiles: {len(profiles_df):,}")
mental_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Dataset info
print("=" * 50)
print("MENTAL HEALTH DATASET INFO")
print("=" * 50)
print(mental_df.info())
print("\n" + "=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
mental_df.describe()

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk label distribution
risk_counts = mental_df['risk_label'].value_counts()
colors = {'LOW': '#4CAF50', 'MEDIUM': '#FFC107', 'HIGH': '#F44336'}
risk_counts.plot(kind='bar', ax=axes[0], color=[colors.get(x, '#666') for x in risk_counts.index])
axes[0].set_title('Mental Health Risk Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# PHQ-9 distribution by risk
for label in ['LOW', 'MEDIUM', 'HIGH']:
    subset = mental_df[mental_df['risk_label'] == label]
    axes[1].hist(subset['phq9_score'], alpha=0.6, label=label, color=colors[label], bins=15)
axes[1].set_title('PHQ-9 Score Distribution by Risk Level')
axes[1].set_xlabel('PHQ-9 Score')
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORT_DIR / 'mental_health_eda.png', dpi=150)
plt.show()

In [ ]:
# Feature correlation heatmap
numeric_cols = ['phq9_score', 'gad7_score', 'mood_score', 'stress_level', 'social_support_score']
plt.figure(figsize=(8, 6))
sns.heatmap(mental_df[numeric_cols].corr(), annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'mental_health_correlation.png', dpi=150)
plt.show()

## 4. Feature Engineering

In [ ]:
# Merge with profile data for additional features
df = mental_df.merge(profiles_df[['user_id', 'age', 'pregnancy_week', 'trimester', 'previous_pregnancies']], 
                     on='user_id', how='left')

# Feature engineering
df['phq9_gad7_combined'] = df['phq9_score'] + df['gad7_score']  # Combined anxiety-depression score
df['mood_stress_ratio'] = df['mood_score'] / (df['stress_level'] + 1)  # Mood-stress balance
df['support_deficit'] = 5 - df['social_support_score']  # Inverse of support (higher = worse)
df['depression_severity'] = pd.cut(df['phq9_score'], bins=[0, 4, 9, 14, 19, 27], 
                                   labels=[0, 1, 2, 3, 4]).astype(float)  # PHQ-9 clinical severity
df['anxiety_severity'] = pd.cut(df['gad7_score'], bins=[0, 4, 9, 14, 21],
                                labels=[0, 1, 2, 3]).astype(float)  # GAD-7 clinical severity

# Encode trimester
trimester_map = {'first': 1, 'second': 2, 'third': 3}
df['trimester_num'] = df['trimester'].map(trimester_map).fillna(2)

print(f"Engineered features added. Total features: {len(df.columns)}")
df.head()

## 5. Prepare Training Data

In [ ]:
# Define features
FEATURE_COLS = [
    'phq9_score', 'gad7_score', 'mood_score', 'stress_level', 'social_support_score',
    'phq9_gad7_combined', 'mood_stress_ratio', 'support_deficit',
    'depression_severity', 'anxiety_severity',
    'age', 'pregnancy_week', 'trimester_num', 'previous_pregnancies'
]

# Prepare X and y
X = df[FEATURE_COLS].copy()
y = df['risk_label'].copy()

# Handle missing values
X = X.fillna(X.median())

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(['LOW', 'MEDIUM', 'HIGH'])  # Explicit order
y_encoded = label_encoder.transform(y)

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nClass distribution:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_encoded == i).sum()} ({(y_encoded == i).mean()*100:.1f}%)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {len(X_train_balanced)} samples")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_train_balanced == i).sum()}")

## 6. Model Training

In [ ]:
# XGBoost with hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Base model
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

# Reduced grid for faster training (use full grid for production)
reduced_grid = {
    'max_depth': [5, 7],
    'n_estimators': [200, 300],
    'learning_rate': [0.1],
    'min_child_weight': [1, 3],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

print("Starting GridSearchCV...")
grid_search = GridSearchCV(
    xgb_model, reduced_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_balanced, y_train_balanced)

In [ ]:
# Best model
best_model = grid_search.best_estimator_
print("\n" + "=" * 50)
print("BEST PARAMETERS")
print("=" * 50)
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV F1 Score: {grid_search.best_score_:.4f}")

## 7. Model Evaluation

In [ ]:
# Predictions
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# AUC-ROC (One-vs-Rest)
try:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
except:
    auc_roc = 0.0

print("\n" + "=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  AUC-ROC:   {auc_roc:.4f}")

In [ ]:
# Classification report
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Mental Health Risk - Confusion Matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'mental_health_confusion_matrix.png', dpi=150)
plt.show()

## 8. Feature Importance & SHAP Analysis

In [ ]:
# Feature importance from XGBoost
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance - Mental Health Model')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'mental_health_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SHAP values
print("Computing SHAP values (this may take a moment)...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled[:100])  # Use subset for speed

# SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=FEATURE_COLS, 
                  class_names=label_encoder.classes_, show=False)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'mental_health_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model Artifacts

In [ ]:
# Save model, scaler, and encoder
joblib.dump(best_model, MODEL_DIR / 'mental_health_xgb.joblib')
joblib.dump(scaler, MODEL_DIR / 'mental_health_scaler.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'mental_health_label_encoder.joblib')

print("✅ Model artifacts saved:")
print(f"   - {MODEL_DIR / 'mental_health_xgb.joblib'}")
print(f"   - {MODEL_DIR / 'mental_health_scaler.joblib'}")
print(f"   - {MODEL_DIR / 'mental_health_label_encoder.joblib'}")

In [ ]:
# Save evaluation metrics
metrics = {
    'mental_health': {
        'model': 'XGBoost',
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'auc_roc': round(auc_roc, 4),
        'best_params': grid_search.best_params_,
        'feature_columns': FEATURE_COLS
    }
}

# Load existing or create new
metrics_file = REPORT_DIR / 'evaluation_metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        all_metrics = json.load(f)
    all_metrics.update(metrics)
else:
    all_metrics = metrics

with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"\n✅ Metrics saved to {metrics_file}")

## 10. Model Inference Test

In [ ]:
# Test inference
def predict_mental_risk(phq9, gad7, mood, stress, social_support, age=30, pregnancy_week=20, 
                        trimester=2, previous_pregnancies=0):
    """Predict mental health risk from input features."""
    # Load artifacts
    model = joblib.load(MODEL_DIR / 'mental_health_xgb.joblib')
    scaler = joblib.load(MODEL_DIR / 'mental_health_scaler.joblib')
    encoder = joblib.load(MODEL_DIR / 'mental_health_label_encoder.joblib')
    
    # Engineered features
    phq9_gad7_combined = phq9 + gad7
    mood_stress_ratio = mood / (stress + 1)
    support_deficit = 5 - social_support
    depression_severity = min(4, phq9 // 5)  # Simplified severity
    anxiety_severity = min(3, gad7 // 5)
    
    features = np.array([[
        phq9, gad7, mood, stress, social_support,
        phq9_gad7_combined, mood_stress_ratio, support_deficit,
        depression_severity, anxiety_severity,
        age, pregnancy_week, trimester, previous_pregnancies
    ]])
    
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    risk_label = encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    return {
        'risk_level': risk_label,
        'confidence': round(confidence, 3),
        'probabilities': {
            cls: round(prob, 3) 
            for cls, prob in zip(encoder.classes_, probabilities)
        }
    }

# Test cases
print("\n" + "=" * 50)
print("INFERENCE TEST")
print("=" * 50)

# Low risk profile
result = predict_mental_risk(phq9=3, gad7=2, mood=8, stress=2, social_support=5)
print(f"\nLow Risk Input: PHQ9=3, GAD7=2, Mood=8, Stress=2, Support=5")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# Medium risk profile
result = predict_mental_risk(phq9=12, gad7=10, mood=5, stress=6, social_support=3)
print(f"\nMedium Risk Input: PHQ9=12, GAD7=10, Mood=5, Stress=6, Support=3")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# High risk profile
result = predict_mental_risk(phq9=22, gad7=18, mood=2, stress=9, social_support=1)
print(f"\nHigh Risk Input: PHQ9=22, GAD7=18, Mood=2, Stress=9, Support=1")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

---

## Summary

✅ **Mental Health Risk Model trained successfully!**

| Metric | Value |
|--------|-------|
| Algorithm | XGBoost |
| Features | 14 (including engineered) |
| Target | LOW / MEDIUM / HIGH |

### Model Artifacts Saved
- `mental_health_xgb.joblib` — Trained XGBoost model
- `mental_health_scaler.joblib` — StandardScaler for feature normalization
- `mental_health_label_encoder.joblib` — LabelEncoder for risk labels

---

⚠️ **Disclaimer**: This model predicts risk likelihood only — NOT a medical diagnosis. Always consult healthcare professionals.